<a href="https://colab.research.google.com/github/nig414/AML-experiments/blob/main/AML_Experiment_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules
import warnings

# 1. Setup & Clean Output
# This hides the 'datetime' deprecation warnings you were seeing
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# 2. Load the Dataset
# Using a direct raw link to the Online Retail CSV on GitHub
url = "https://raw.githubusercontent.com/StephanT324/Market_Basket_Analysis/master/OnlineRetail.csv"

print("Fetching data from verified GitHub mirror...")
try:
    # Reading with 'latin1' encoding to handle special characters in product names
    df = pd.read_csv(url, encoding='latin1')
    print("✓ Data loaded successfully!")

    # 3. Data Cleaning
    print("Processing and cleaning...")
    df['Description'] = df['Description'].str.strip()
    df.dropna(axis=0, subset=['InvoiceNo'], inplace=True)
    df['InvoiceNo'] = df['InvoiceNo'].astype('str')

    # Remove credit transactions (Invoice numbers starting with 'C')
    df = df[~df['InvoiceNo'].str.contains('C')]

    # 4. Create the 'Basket' (Using France as a sample for performance)
    basket = (df[df['Country'] =="France"]
              .groupby(['InvoiceNo', 'Description'])['Quantity']
              .sum().unstack().reset_index().fillna(0)
              .set_index('InvoiceNo'))

    # 5. One-Hot Encoding
    # The algorithm requires 0 or 1 (Present/Absent)
    def encode_units(x):
        return 1 if x >= 1 else 0

    basket_sets = basket.applymap(encode_units)

    # Drop 'POSTAGE' as it's a shipping fee, not a product
    if 'POSTAGE' in basket_sets.columns:
        basket_sets.drop('POSTAGE', inplace=True, axis=1)

    # 6. Apply Apriori Algorithm
    # min_support=0.07 means the item appears in at least 7% of orders
    frequent_itemsets = apriori(basket_sets, min_support=0.07, use_colnames=True)

    # 7. Generate Association Rules
    # We use 'lift' to find the strongest relationships
    rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)
    rules = rules.sort_values(['lift', 'confidence'], ascending=[False, False])

    # 8. Output Top 10 Results
    print("\n--- Top 10 Product Associations found in France ---")
    print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

    # 9. Visualization
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=rules['support'], y=rules['confidence'],
                    size=rules['lift'], hue=rules['lift'], palette='plasma')
    plt.title('Market Basket Analysis: Support vs. Confidence')
    plt.xlabel('Support (How common the pair is)')
    plt.ylabel('Confidence (Probability of buying B if A is bought)')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

except Exception as e:
    print(f"\n[ERROR]: {e}")
    print("If you still see a 404, please try running: !wget {url} to check connectivity.")

Fetching data from verified GitHub mirror...

[ERROR]: HTTP Error 404: Not Found
If you still see a 404, please try running: !wget {url} to check connectivity.


In [ ]:
!pip install mlxtend --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 82.7 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninst